# 🚀 TACYOLO: 3TB Streaming Batch-Tiled Training on Google Colab & Google Drive

This notebook trains TACYOLO across multi-terabyte tactical, aerial, and thermal datasets directly on **Google Drive** without filling local storage.

### Core Pipeline Features:
1. **Direct-to-Drive Storage**: Staged completely inside `/content/drive/MyDrive/TACYOLO/` (`working_testing`, `raw_data`, `tiled_batches`, `trained_weights`).
2. **Small-Object Tiling**: High-resolution drone/satellite imagery is partitioned into overlapping 640x640 tiles with bounding-box re-projection.
3. **Streaming Auto-Purge**: Trains in manageable batches, checkpoints the model, and automatically purges processed raw downloads and tiles to recycle disk quota.
4. **INT8 Edge Calibration**: Generates a TensorRT calibration cache from local operational footage with zero accuracy loss.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Set up the prescribed folder hierarchy
import os
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/TACYOLO')
WORKING_TESTING = DRIVE_ROOT / 'working_testing'
RAW_DATA = DRIVE_ROOT / 'raw_data'
TILED_BATCHES = DRIVE_ROOT / 'tiled_batches'
TRAINED_WEIGHTS = DRIVE_ROOT / 'trained_weights'
CALIBRATION_DATA = DRIVE_ROOT / 'calibration_data'
METRICS_LOGS = DRIVE_ROOT / 'metrics_logs'

for folder in [DRIVE_ROOT, WORKING_TESTING, RAW_DATA, TILED_BATCHES, TRAINED_WEIGHTS, CALIBRATION_DATA, METRICS_LOGS]:
    folder.mkdir(parents=True, exist_ok=True)
    print(f"Ready: {folder}")


## 2. Setup Kaggle Credentials
Provide your Kaggle API credentials to download tactical datasets directly to Drive.

In [ ]:
# Option A: Use Colab Secrets / Environment Variables
# import os
# os.environ['KAGGLE_USERNAME'] = 'your_username'
# os.environ['KAGGLE_KEY'] = 'your_api_key'

# Option B: Upload kaggle.json
from google.colab import files
import shutil

kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(parents=True, exist_ok=True)
kaggle_json = kaggle_dir / 'kaggle.json'

if not kaggle_json.exists():
    print("Upload your kaggle.json file:")
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn.endswith('kaggle.json'):
            shutil.move(fn, str(kaggle_json))
            os.chmod(str(kaggle_json), 0o600)
            print("Kaggle token installed successfully.")
else:
    print("Kaggle credentials already present.")


## 3. Install Dependencies & Clone TACYOLO

In [ ]:
!pip install -q ultralytics kagglehub opencv-python onnx onnxruntime

# If running directly from git repository:
!git clone https://github.com/devansh3108-2/tacyolo.git /content/tacyolo_repo || (cd /content/tacyolo_repo && git pull)
%cd /content/tacyolo_repo
!pip install -e .


## 4. Launch 3TB Streaming Batch-Tiled Training Pipeline
Each batch:
1. Downloads the Kaggle dataset directly to Google Drive `raw_data/`.
2. Slices the high-res aerial/thermal frames into 640x640 tiles in `tiled_batches/`.
3. Fine-tunes the YOLO model, continuously checkpointing `best.pt` to `trained_weights/`.
4. Automatically purges raw downloads and tiles to keep storage under Drive quota.

In [ ]:
!python scripts/colab_tile_train.py \
    --drive-root /content/drive/MyDrive/TACYOLO \
    --tile-size 640 \
    --epochs 15


## 5. Harden INT8 Quantization & Calibration
Build a TensorRT calibration cache using operational tactical footage directly from Drive.

In [ ]:
!python -m tacyolo.runtime.quantize \
    --weights /content/drive/MyDrive/TACYOLO/trained_weights/best.pt \
    --calib-data /content/drive/MyDrive/TACYOLO/calibration_data \
    --output-dir /content/drive/MyDrive/TACYOLO/trained_weights \
    --imgsz 640


## 6. Verify Model Artifacts on Google Drive

In [ ]:
!ls -lh /content/drive/MyDrive/TACYOLO/trained_weights
